# scvi/03 — Clonotype Integration & Exhaustion Scoring (Python)

Python port of `03_clonotype_exhaustion.ipynb`.  
Reads `data/scvi/query_projected.h5ad` (output of scvi/02).

**Steps:**
1. Load scVI-projected query AnnData
2. Subset CD8+ T cells
3. Assign / simulate TCR clonotypes
4. Calculate exhaustion score via `sc.tl.score_genes` (Tirosh 2016)
5. Aggregate to clonotype level
6. Rank and select top exhausted clonotypes
7. Save outputs

**Inputs:** `data/scvi/query_projected.h5ad`, `data/exhaustion_gene_panel.txt`  
**Outputs:** `data/query_with_clonotypes.h5ad`, `data/ranked_clonotypes_scvi.csv`

In [1]:
query_h5ad              = "data/scvi/query_projected.h5ad"
exhaustion_panel_path   = "data/exhaustion_gene_panel.txt"
output_clonotype_table  = "data/scvi/ranked_clonotypes.csv"
output_h5ad             = "data/scvi/query_with_clonotypes.h5ad"
min_cells_per_clonotype = 2
n_exhausted_clonotypes  = 20
random_seed             = 42
labels_key              = "predicted_label"
cd8_pattern             = "cd8"


In [2]:
# Parameters
query_h5ad = "data/scvi/query_projected.h5ad"
exhaustion_panel_path = "data/exhaustion_gene_panel.txt"
output_clonotype_table = "data/scvi/ranked_clonotypes.csv"
output_h5ad = "data/scvi/query_with_clonotypes.h5ad"
min_cells_per_clonotype = 2
n_exhausted_clonotypes = 20


In [3]:
import os, time
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.random.seed(random_seed)
t_start = time.time()
print(f"anndata {ad.__version__} | scanpy {sc.__version__}")


anndata 0.12.11 | scanpy 1.11.5


/var/folders/8y/ph922wl553g48t1wthjdsfnm0000gn/T/ipykernel_21056/2585432606.py:12: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(f"anndata {ad.__version__} | scanpy {sc.__version__}")
/var/folders/8y/ph922wl553g48t1wthjdsfnm0000gn/T/ipykernel_21056/2585432606.py:12: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"anndata {ad.__version__} | scanpy {sc.__version__}")


## 1 · Load Projected Query

In [4]:
adata = ad.read_h5ad(query_h5ad)
print(f"Loaded {query_h5ad}: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"obs columns : {list(adata.obs.columns)}")
print(f"obsm keys   : {list(adata.obsm.keys())}")

if labels_key not in adata.obs.columns:
    fallback = next((c for c in adata.obs.columns
                     if "cluster" in c.lower() or "label" in c.lower()), None)
    print(f"Warning: '{labels_key}' not found — using '{fallback}'")
    labels_key = fallback

print(f"\nCell type distribution ('{labels_key}'):")
print(adata.obs[labels_key].value_counts().to_string())


Loaded data/scvi/query_projected.h5ad: 30,683 cells x 2,000 genes
obs columns : ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'patient', 'treatment', 'cluster', 'UMAP1', 'UMAP2', 'clonotype_id', 'has_tcr', 'percent.mt', 'RNA_snn_res.0.5', 'seurat_clusters', 'doublet_score', 'predicted_doublet', '_scvi_batch', '_scvi_labels', 'predicted_label', 'is_cd8']
obsm keys   : ['X_scANVI', 'X_umap']

Cell type distribution ('predicted_label'):
predicted_label
Th17          6753
CD8_mem       5435
Naive         4757
Tregs         4259
CD8_act       3761
Tfh           2834
CD8_ex        1514
CD8_eff        951
CD8_ex_act     419


## 2 · Subset to CD8+ T Cells

In [5]:
if "is_cd8" in adata.obs.columns:
    cd8_mask = adata.obs["is_cd8"].astype(bool)
    print("Using 'is_cd8' column from scvi/02")
else:
    cd8_mask = adata.obs[labels_key].str.contains(cd8_pattern, case=False, na=False)
    print(f"Pattern-matched '{cd8_pattern}' in '{labels_key}'")

cd8 = adata[cd8_mask].copy()
print(f"CD8+ T cells: {cd8.n_obs:,} / {adata.n_obs:,} ({cd8.n_obs/adata.n_obs*100:.1f}%)")
print(cd8.obs[labels_key].value_counts().to_string())


Using 'is_cd8' column from scvi/02


CD8+ T cells: 12,080 / 30,683 (39.4%)
predicted_label
CD8_mem       5435
CD8_act       3761
CD8_ex        1514
CD8_eff        951
CD8_ex_act     419


## 3 · Clonotype Assignments

Uses existing `clonotype_id` from obs (10x VDJ / scRepertoire) if present; otherwise simulates power-law clonotypes.

In [6]:
rng = np.random.default_rng(random_seed)
aa  = list("ACDEFGHIKLMNPQRSTVWY")

if "clonotype_id" in cd8.obs.columns and cd8.obs["clonotype_id"].notna().any():
    print("Using existing clonotype_id from obs")
    clono_ids = cd8.obs["clonotype_id"].dropna().unique()
else:
    print("No clonotype_id found — simulating power-law clonotypes")
    n_cells      = cd8.n_obs
    n_clonotypes = max(20, n_cells // 4)
    ranks        = np.arange(1, n_clonotypes + 1)
    probs        = ranks ** -1.8
    probs       /= probs.sum()
    clono_ids    = np.array([f"clonotype_{i:03d}" for i in ranks])
    cd8.obs["clonotype_id"] = np.random.choice(clono_ids, size=n_cells, p=probs)

tcr_ref = pd.DataFrame({
    "clonotype_id": clono_ids,
    "TRAV": [f"TRAV{rng.integers(1,40)}-{rng.integers(1,6)}" for _ in clono_ids],
    "TRBV": [f"TRBV{rng.integers(1,31)}-{rng.integers(1,6)}" for _ in clono_ids],
    "CDR3a": ["CASS" + "".join(rng.choice(aa, rng.integers(8,14))) for _ in clono_ids],
    "CDR3b": ["CASS" + "".join(rng.choice(aa, rng.integers(8,14))) for _ in clono_ids],
})

clone_counts = cd8.obs["clonotype_id"].value_counts()
print(f"Unique clonotypes: {clone_counts.shape[0]}")
print(f"Top 10:\n{clone_counts.head(10).to_string()}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(clone_counts.values, bins=30, color="steelblue", edgecolor="white")
ax.set_xlabel("Clone size (# cells)")
ax.set_ylabel("# clonotypes")
ax.set_title("Clone size distribution — CD8+ T cells")
plt.tight_layout(); plt.show()


Using existing clonotype_id from obs
Unique clonotypes: 3710
Top 10:
clonotype_id
CATKGYQAGELFF        365
CASSIDWTGYLDTQYF      96
CASSLNPGADTQYF        94
CASSQEGQSSYNSPLHF     75
CASSIDWTGYLQPQHF      74
CASSQTSGIYNEQFF       73
CASSLVVADPYQETQYF     58
CSARDRGYGNTIYF        58
CATSDLGQGVGNEQFF      55
CASSQEGSSGYEQYF       54


/var/folders/8y/ph922wl553g48t1wthjdsfnm0000gn/T/ipykernel_21056/2311293448.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4 · Exhaustion Score — `sc.tl.score_genes`

`scanpy.tl.score_genes` implements the Tirosh et al. (2016) method, equivalent to Seurat `AddModuleScore`.

In [7]:
with open(exhaustion_panel_path) as fh:
    panel_genes = [l.strip() for l in fh if l.strip() and not l.startswith("#")]
print(f"Exhaustion panel ({len(panel_genes)} genes): {' '.join(panel_genes)}")

present = [g for g in panel_genes if g in cd8.var_names]
absent  = [g for g in panel_genes if g not in cd8.var_names]
print(f"Present: {len(present)} | Absent: {len(absent)}")
if absent:
    print(f"Absent: {', '.join(absent)}")

if len(present) >= 3:
    sc.tl.score_genes(cd8, gene_list=present, score_name="exhaustion_score",
                      ctrl_size=20, random_state=random_seed)
    print("sc.tl.score_genes complete")
else:
    print("WARNING: < 3 panel genes present — using simulated scores")
    cd8.obs["exhaustion_score"] = np.random.normal(0.3, 0.6, cd8.n_obs)

print(cd8.obs["exhaustion_score"].describe().to_string())


Exhaustion panel (18 genes): PDCD1 HAVCR2 LAG3 TIGIT CTLA4 TOX NR4A1 ENTPD1 CXCL13 PRDM1 EOMES TBX21 GZMB PRF1 IFNG TNF CD38 VCAM1
Present: 17 | Absent: 1
Absent: TBX21


sc.tl.score_genes complete
count    12080.000000
mean        -0.181590
std          0.962053
min         -4.698529
25%         -0.617647
50%         -0.323529
75%          0.014706
max         19.588235


In [8]:
fig, axes = plt.subplots(1, 2 if "X_umap" in cd8.obsm else 1,
                         figsize=(12 if "X_umap" in cd8.obsm else 5, 4))
ax_hist = axes[0] if isinstance(axes, np.ndarray) else axes
ax_hist.hist(cd8.obs["exhaustion_score"], bins=40, color="salmon", edgecolor="white")
ax_hist.set_xlabel("Exhaustion score"); ax_hist.set_ylabel("# cells")
ax_hist.set_title("Per-cell exhaustion score")

if "X_umap" in cd8.obsm:
    umap = cd8.obsm["X_umap"]
    sc_plot = axes[1].scatter(umap[:, 0], umap[:, 1],
        c=cd8.obs["exhaustion_score"], cmap="magma", s=3, alpha=0.7)
    plt.colorbar(sc_plot, ax=axes[1], label="Exhaustion score")
    axes[1].set_title("Exhaustion on scANVI UMAP")
    axes[1].set_xlabel("UMAP1"); axes[1].set_ylabel("UMAP2")

plt.tight_layout(); plt.show()


/var/folders/8y/ph922wl553g48t1wthjdsfnm0000gn/T/ipykernel_21056/38161949.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5 · Aggregate to Clonotype Level

| Column | Definition |
|--------|------------|
| `clone_size` | # cells in clonotype |
| `mean_exhaustion_score` | mean per-cell exhaustion score |
| `clone_frequency` | clone_size / total CD8 cells |
| `rank_score` | `mean_exhaustion × log1p(clone_size)` |

In [9]:
agg = (
    cd8.obs
    .groupby("clonotype_id")["exhaustion_score"]
    .agg(clone_size="count", mean_exhaustion_score="mean", sd_exhaustion_score="std")
    .reset_index()
)
agg["clone_frequency"] = agg["clone_size"] / cd8.n_obs
agg["rank_score"]      = agg["mean_exhaustion_score"] * np.log1p(agg["clone_size"])

clono_df = (
    agg
    .merge(tcr_ref, on="clonotype_id", how="left")
    .query("clone_size >= @min_cells_per_clonotype")
    .sort_values("rank_score", ascending=False)
    .reset_index(drop=True)
)
clono_df["rank"] = clono_df.index + 1

print(f"Clonotypes after size filter (>= {min_cells_per_clonotype}): {len(clono_df)}")
print(clono_df[["clonotype_id","clone_size","mean_exhaustion_score",
               "clone_frequency","rank_score","CDR3b"]].head(10).to_string(index=False))


Clonotypes after size filter (>= 2): 1102
       clonotype_id  clone_size  mean_exhaustion_score  clone_frequency  rank_score             CDR3b
      CATKGYQAGELFF         365               1.699033         0.030215   10.028769 CASSEAKCCLIDYFFTR
     CASSSTGANYGYTF           2               8.345588         0.000166    9.168566      CASSMCAHNRYQ
   CASSPNVGRGNYGYTF          10               3.463235         0.000828    8.304476    CASSPPFTKTTAAM
   CASSLGRGSGANVLTF           6               4.148284         0.000497    8.072189     CASSLYKTQVHTT
CASSYFIRRTGGHQETQYF           9               2.830882         0.000745    6.518348 CASSLTLDDVPDINLCM
      CASSPFINTEAFF           7               2.945378         0.000579    6.124742      CASSSWERNVEH
     CASSQDILNTEAFF           2               5.191176         0.000166    5.703090   CASSDSLNRQACTVG
      CASSLPTTGELFF          23               1.759591         0.001904    5.592074 CASSYVKRGNGPKICGK
     CASSHLSAGYEQFF          34         

/var/folders/8y/ph922wl553g48t1wthjdsfnm0000gn/T/ipykernel_21056/3599867719.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("clonotype_id")["exhaustion_score"]


In [10]:
top_clonotypes = clono_df.head(n_exhausted_clonotypes)
print(f"Top {n_exhausted_clonotypes} exhausted clonotypes selected")
print(f"Mean clone size : {top_clonotypes['clone_size'].mean():.1f}")
print(f"Mean exhaustion : {top_clonotypes['mean_exhaustion_score'].mean():.3f}")


Top 20 exhausted clonotypes selected
Mean clone size : 29.1
Mean exhaustion : 2.809


## 6 · Visualise Clonotype Landscape

In [11]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

is_top = clono_df["rank"] <= n_exhausted_clonotypes
ax1.scatter(clono_df.loc[~is_top, "clone_frequency"],
            clono_df.loc[~is_top, "mean_exhaustion_score"],
            s=clono_df.loc[~is_top, "clone_size"] * 2,
            c="#cccccc", alpha=0.6, label="other")
ax1.scatter(clono_df.loc[is_top, "clone_frequency"],
            clono_df.loc[is_top, "mean_exhaustion_score"],
            s=clono_df.loc[is_top, "clone_size"] * 2,
            c="#e63946", alpha=0.8, label=f"top {n_exhausted_clonotypes}")
ax1.set_xlabel("Clone frequency"); ax1.set_ylabel("Mean exhaustion score")
ax1.set_title("Clonotype landscape"); ax1.legend()

top20 = clono_df.head(min(20, len(clono_df)))
ax2.barh(top20["clonotype_id"][::-1], top20["rank_score"][::-1],
         color="#e63946", alpha=0.85)
ax2.set_xlabel("Rank score")
ax2.set_title(f"Top {min(20, n_exhausted_clonotypes)} exhausted clonotypes")

plt.tight_layout()
plt.savefig("data/clonotype_landscape.png", dpi=120)
plt.show()


/var/folders/8y/ph922wl553g48t1wthjdsfnm0000gn/T/ipykernel_21056/4068604051.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 · Save Outputs

In [12]:
os.makedirs(os.path.dirname(output_clonotype_table) or ".", exist_ok=True)

clono_df.to_csv(output_clonotype_table, index=False)
print(f"Ranked clonotype table -> {output_clonotype_table}")

rank_map = dict(zip(clono_df["clonotype_id"], clono_df["rank"]))
cd8.obs["clonotype_rank"]   = cd8.obs["clonotype_id"].map(rank_map)
cd8.obs["is_top_exhausted"] = cd8.obs["clonotype_rank"] <= n_exhausted_clonotypes

cd8.write_h5ad(output_h5ad)
print(f"Annotated CD8 AnnData -> {output_h5ad}")

t_total = time.time() - t_start
print(f"\nSummary:")
print(f"  CD8+ T cells              : {cd8.n_obs:,}")
print(f"  Clonotypes (>= {min_cells_per_clonotype} cells)  : {len(clono_df)}")
print(f"  Top exhausted selected    : {n_exhausted_clonotypes}")
print(f"  Total time                : {t_total:.1f}s")


Ranked clonotype table -> data/scvi/ranked_clonotypes.csv
Annotated CD8 AnnData -> data/scvi/query_with_clonotypes.h5ad

Summary:
  CD8+ T cells              : 12,080
  Clonotypes (>= 2 cells)  : 1102
  Top exhausted selected    : 20
  Total time                : 0.7s
